In [2]:
homedir = '/mnt/mirabelle/az6922_homedir/DRing/src/emp/datacentre/'
import random
import numpy as np

# from makec2s.ipynb
def genflowbytes():
    np.random.seed(0)
    
    mean_bytes = 100.0 * 1024
    shape = 1.05
    scale = mean_bytes * (shape - 1)/shape

    x = np.random.exponential(scale=1.0/shape)
    flowbytes = int(scale * np.exp(x))
    return flowbytes

def adjustbytesbymtu(flowbytes):
  mss = 1500
  return mss * ((flowbytes+mss-1)//mss)

large_flow_threshold = 10 * 1024 * 1024

In [3]:
p = 40
a = 2
h = 19

korn =2 #su2

topologyfile = f"evaltopologyfiles/df_p{p}_a{a}_h{h}.edgelist"
serverfile = f"evalserverfiles/df_p{p}_a{a}_h{h}.sv"
numsw = a*(a*h+1)
npfile = f"evalnetpathfiles/netpath_df_p{p}_a{a}_h{h}_su{korn}.np"

In [ ]:
# # current dir: ~/DRing/src/emp/datacentre/

# # generate topology files
# print(f"python3 generate_dragonfly_topologyfiles_serverfiles.py --p {p} --a {a} --h {h} --topologyfile {topologyfile} --serverfile {serverfile}")

# # generate netpath files
# print(f"python3 generate_suk_netpathfiles.py --numsw {numsw} --korn {korn} --graphfile {topologyfile} --netpathfile {npfile}")

python3 generate_dragonfly_topologyfiles_serverfiles.py --p 40 --a 2 --h 19 --topologyfile evaltopologyfiles/df_p40_a2_h19.edgelist --serverfile evalserverfiles/df_p40_a2_h19.sv
python3 generate_suk_netpathfiles.py --numsw 78 --korn 2 --graphfile evaltopologyfiles/df_p40_a2_h19.edgelist --netpathfile evalnetpathfiles/netpath_df_p40_a2_h19_su2.np


In [6]:
stime = 144 # ms
g = a*h+1
nlinks = a*(a-1)*g + g*(g-1) # uni-directional
nhosts = a*p*g
bw = 1342176000 # B per second
load_list = range(1,11)
seed_list = [1,2,3,4,5]
topologytype = 3
nswitches = a*g
k = p+h+a-1 # nports
os = 1 #unused
nintervals = 8

print(f"#switches: {numsw}, #servers: {nhosts}, #links: {nlinks}, #groups: {g}, #ports: {k}")

#switches: 78, #servers: 3120, #links: 1560, #groups: 39, #ports: 60


In [5]:
# generate connection_matrices file (1)
prv1bytes = 0
prv1file = f'{homedir}rawtrafficfiles/prv1'
maxinterval = 0
with open(prv1file, 'r') as f:
    lines = f.readlines()
    for line in lines:
        tokens = line.split(',')
        # 0,32,31,10500
        # interval,fromserver,toserver,bytes
        prv1bytes += int(tokens[3])
        maxinterval = max(maxinterval, int(tokens[0]))
print(f'prv1bytes {prv1bytes}, maxinterval {maxinterval}, fullload {bw * stime * nlinks / 1000}, ratio {(bw * stime * nlinks / 1000) / prv1bytes}')

prv1bytes 222240244500, maxinterval 7, fullload 301506416640.0, ratio 1.3566688486972034


In [7]:
# generate connection_matrices file (2)
random.seed(0)
for load in load_list:
    totalbytes = bw * stime / 1000 * nlinks * load / 100  # B
    mult = totalbytes / prv1bytes
    actualbytes = 0
    cmfile = f'cmfiles/df2_load{load}.cm'
    with open(cmfile, 'w') as fw:
        with open(prv1file, 'r') as fr:
            lines = fr.readlines()
            iline = 0
            while actualbytes < totalbytes:
                line = lines[iline]
                tokens = line.split(',')
                interval = int(tokens[0])
                fromserver = int(tokens[1])
                toserver = int(tokens[2])
                multbytes = int(tokens[3])

                if fromserver >= nhosts or toserver >= nhosts:
                    iline += 1
                    if iline >= len(lines):
                        iline = 0
                        if mult-1>0:
                            mult = mult-1
                    continue

                if mult >= 1 or (random.random() < mult):
                    multbytes = adjustbytesbymtu(multbytes)
    
                    # generate random start time
                    start_time_ms = random.uniform(0, stime//(maxinterval+1)) + interval * (stime//(maxinterval+1))

                    fw.write(f'{fromserver},{toserver},{int(multbytes)},{start_time_ms:.4f}\n')
                    actualbytes += int(multbytes)

                iline += 1
                if iline >= len(lines):
                    iline = 0
                    if mult-1>0:
                        mult = mult-1

                    # print(f'actualbytes {actualbytes}, totalbytes {totalbytes}, mult {mult}', end='\r')

    print(f'load {load}%, totalbytes {totalbytes}, prv1bytes {prv1bytes}, mult {mult}, actualbytes {actualbytes}')


load 1%, totalbytes 3015064166.4, prv1bytes 222240244500, mult 0.013566688486972034, actualbytes 3015064500
load 2%, totalbytes 6030128332.8, prv1bytes 222240244500, mult 0.02713337697394407, actualbytes 6030154500
load 3%, totalbytes 9045192499.2, prv1bytes 222240244500, mult 0.040700065460916104, actualbytes 9045328500
load 4%, totalbytes 12060256665.6, prv1bytes 222240244500, mult 0.05426675394788814, actualbytes 12061372500
load 5%, totalbytes 15075320832.0, prv1bytes 222240244500, mult 0.06783344243486017, actualbytes 15075324000
load 6%, totalbytes 18090384998.4, prv1bytes 222240244500, mult 0.08140013092183221, actualbytes 18091396500
load 7%, totalbytes 21105449164.8, prv1bytes 222240244500, mult 0.09496681940880423, actualbytes 21105463500
load 8%, totalbytes 24120513331.2, prv1bytes 222240244500, mult 0.10853350789577627, actualbytes 24120519000
load 9%, totalbytes 27135577497.6, prv1bytes 222240244500, mult 0.1221001963827483, actualbytes 27135586500
load 10%, totalbytes 301

In [8]:
# generate pathweight file (1)
interval_stime = stime / nintervals
with open('df2su2_generate_pwfiles.conf', 'w') as f:
    for load in load_list:
        cmfile = f'cmfiles/df2_load{load}.cm'
        for interval in range(nintervals):
            flowstart = interval_stime * interval
            flowend = interval_stime * (interval + 1)
            varfile = f'{homedir}rawpathweightfiles/pathtraffic_df2_{nhosts}_{nswitches}_{k}_su2_prv1_load{load}_interval{interval}.var'
            qvarfile = f'{homedir}rawpathweightfiles/pathweight_df2_{nhosts}_{nswitches}_{k}_su2_prv1_load{load}_interval{interval}.var'
            f.write(f"python3 {homedir}generate_pathweightfiles.py --graphfile {homedir}{topologyfile} --serverfile {homedir}{serverfile} --numsw {nswitches} --numserver {nhosts} --netpathfile {homedir}{npfile} --flowfile {cmfile} --flowstart {flowstart} --flowend {flowend} --numfaillink 0 --linkfailurefile none --varfile {varfile} --qvarfile {qvarfile}\n")

(current dir: ~/DRing/src/emp/datacentre/experiments/nsdi26fall/eval_main/prv1/)
python3 ../../../../pararun.py --conf df2su2_generate_pwfiles.conf --worker 100

In [9]:
# generate pathweight file (2)
intervaldict = {0:0,1:0,2:1,3:2,4:3,5:4,6:5,7:6} # to:from
with open('df2su2_copy_pwfiles.conf', 'w') as f:
    for load in load_list:
        for interval in range(nintervals):
            fromfile = f'{homedir}rawpathweightfiles/pathweight_df2_{nhosts}_{nswitches}_{k}_su2_prv1_load{load}_interval{intervaldict[interval]}.var'
            tofile = f'{homedir}experiments/nsdi26fall/eval_main/prv1/pwfiles/pathweight_df2_su2_prv1_load{load}_interval{interval}.pw'
            f.write(f'cp {fromfile} {tofile}\n')

actually run the copy commands in datacentre/

In [10]:
# generate conf file
conffile = f'{homedir}experiments/nsdi26fall/eval_main/prv1/run_df2su2.conf'
with open(conffile, 'w') as f:
    for seed in seed_list:
        for load in load_list:
            cmfile = f'experiments/nsdi26fall/eval_main/prv1/cmfiles/df2_load{load}.cm'
            pwfileprefix = f'experiments/nsdi26fall/eval_main/prv1/pwfiles/pathweight_df2_su2_prv1_load{load}_interval'
            outfile = f'experiments/nsdi26fall/eval_main/prv1/outfiles/df2su2_load{load}_seed{seed}.out'
            f.write(f"./eval -stime {stime} -seed {seed} -df_p {p} -df_a {a} -df_h {h} -cmfile {cmfile} -topologytype {topologytype} -npfile {npfile} -pwfileprefix {pwfileprefix} -numintervals {nintervals} -serverfile {serverfile} -topologyfile {topologyfile} > {outfile}\n")
            

python3 pararun.py --conf experiments/nsdi26fall/eval_main/prv1/run_df2su2.conf --worker 50